# Day 23 - Incremental Watermark and Idempotency

**Topic:** Incremental Watermark and Idempotency  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** ฝึกทำ incremental load ด้วย `watermark`, `MERGE` และ `run log` เพื่อให้ pipeline รันซ้ำได้โดยไม่ทำให้ข้อมูลซ้ำหรือเสีย

วันนี้เราจะต่อจาก pattern ฝั่ง Bronze / Silver / Data Quality / Quarantine ที่ผ่านมา แล้วโฟกัสเรื่อง incremental processing แบบจับต้องได้ใน PySpark Lakehouse: อ่านเฉพาะข้อมูลใหม่หรือข้อมูลที่เปลี่ยน, update target table แบบปลอดภัย, เก็บ watermark หลัง run สำเร็จ และตรวจว่า rerun แล้วข้อมูลไม่ duplicate


## Goal of the day

1. อธิบายได้ว่า **watermark** คืออะไร และใช้ตัด incremental window อย่างไร
2. แยกให้ออกระหว่าง **full load**, **incremental load**, และ **rerun**
3. ใช้ `updated_at` เป็นตัวอย่าง watermark column เพื่อ process เฉพาะ new/changed records
4. ใช้ `MERGE` เพื่อทำ idempotent write เข้า Lakehouse table
5. สร้าง run log แบบง่ายเพื่อเห็นว่าแต่ละ run อ่านกี่ records, merge กี่ records และ update watermark ถึงจุดไหน


## Why it matters in production

ใน production pipeline ส่วนใหญ่เราไม่อยาก scan และประมวลผลข้อมูลทั้งหมดทุกครั้ง เพราะเสียเวลา เสีย cost และเสี่ยงกระทบ downstream โดยไม่จำเป็น

Incremental load ช่วยให้ pipeline ทำงานเฉพาะข้อมูลที่เปลี่ยน แต่จะปลอดภัยจริงก็ต่อเมื่อมี:

- **Watermark** เพื่อรู้ว่าประมวลผลสำเร็จถึงจุดไหนแล้ว
- **Idempotent write** เพื่อให้รันซ้ำแล้วผลลัพธ์ไม่ซ้ำ ไม่ corrupt
- **Run log** เพื่อ audit ได้ว่าแต่ละ run ทำอะไรไปบ้าง
- **Duplicate prevention** เพื่อให้ข้อมูลใน source batch เหลือ record ล่าสุดต่อ business key ก่อน merge
- **Safe watermark update** เพื่อไม่เลื่อน watermark ถ้า write target ยังไม่สำเร็จ

Production takeaway:

> Incremental load ที่ดีไม่ใช่แค่ `filter(updated_at > last_watermark)` แต่ต้องออกแบบทั้ง read logic, write pattern, duplicate handling, run log และ rerun behavior ให้ปลอดภัยด้วย


## Key concepts

| Concept | Meaning |
|---|---|
| Incremental load | ประมวลผลเฉพาะ records ใหม่หรือ records ที่เปลี่ยนจากรอบก่อน |
| Watermark | ค่าที่บอกว่า pipeline process สำเร็จถึงจุดไหนแล้ว เช่น `last_successful_watermark` |
| Watermark column | column ที่ใช้ตัด incremental window เช่น `updated_at`, `modified_at`, `event_time`, `ingestion_time` |
| Lower watermark | จุดเริ่มต้นของรอบนี้ ปกติคือ watermark ล่าสุดที่สำเร็จ |
| Upper watermark | จุดสิ้นสุดของรอบนี้ ปกติคือเวลาที่ pipeline เลือกเป็นขอบเขตของ batch |
| Idempotency | รันซ้ำแล้วได้ผลลัพธ์ปลายทางเหมือนเดิม ไม่ duplicate และไม่ทำให้ target เสีย |
| MERGE / upsert | update ถ้า key มีอยู่แล้ว, insert ถ้า key ยังไม่มี |
| Run log | log ระดับ run เช่น run id, status, source count, merge count, lower/upper watermark |
| Rerun safety | ความสามารถในการ rerun batch โดยไม่ทำให้ข้อมูลซ้ำหรือ watermark ผิด |


## Concept explanation

### Incremental load คืออะไร?

Full load คือการอ่านข้อมูลทั้งหมดทุกครั้ง ส่วน incremental load คือการอ่านเฉพาะ records ที่ใหม่หรือเปลี่ยนจากรอบก่อน เช่น:

```python
updated_at > last_successful_watermark
```

ใน notebook วันนี้ เราจะใช้ `updated_at` เป็นตัวอย่าง watermark column เพราะเข้าใจง่ายและใกล้เคียงงานจริง

### Watermark ต้องเก็บหลัง run สำเร็จ

Watermark ไม่ควรถูก update ทันทีตั้งแต่เริ่ม run เพราะถ้า target write fail แต่ watermark เลื่อนไปแล้ว รอบถัดไปอาจข้ามข้อมูลชุดนั้นไปเลย

Pattern ที่ปลอดภัยกว่า:

1. อ่าน watermark ล่าสุด
2. filter source ด้วย lower/upper watermark
3. deduplicate source batch
4. merge เข้า target table
5. เขียน run log
6. update watermark หลังจาก write สำเร็จแล้วเท่านั้น

### Idempotency คืออะไรในมุม Data Engineering?

Idempotency แปลว่า run เดิมซ้ำแล้ว target table ยังอยู่ใน state ที่ถูกต้อง เช่น:

- ไม่เกิด duplicate records จากการ append ซ้ำ
- business key เดิมถูก update แทนการ insert ซ้ำ
- rerun แล้ว row count ไม่พุ่งผิดปกติ
- watermark ไม่ถอยหลังหรือเลื่อนไปผิดจังหวะ

พูดง่าย ๆ:

> Incremental filter ช่วยลดข้อมูลที่อ่าน  
> Idempotent write ช่วยให้รันซ้ำแล้วไม่พัง

### ทำไมต้องมี upper watermark?

ถ้าใช้แค่ `updated_at > last_watermark` แล้ว source ยังมีข้อมูลไหลเข้าระหว่าง run อาจทำให้ขอบเขต batch ไม่ชัด

ใน lab นี้เราจะกำหนด `upper_watermark` เอง เช่น `2026-05-02 23:59:59` เพื่อให้ batch window ชัดเจน:

```text
lower_watermark < updated_at <= upper_watermark
```

### ข้อจำกัดที่ต้องรู้

Watermark ด้วย `updated_at` แบบตรง ๆ ยังมี edge cases เช่น late-arriving records, timestamp tie (records ที่มี `updated_at` เท่ากัน), timezone mismatch, source clock drift และ deleted records

วันนี้จะยังไม่แก้ทุกเคสแบบ production เต็มรูปแบบ แต่จะทำให้เห็น pattern หลักที่นำไปต่อยอดได้


## Mermaid diagram: Incremental Watermark Flow

```mermaid
flowchart LR
    A[Source records with updated_at] --> B[Read lower watermark]
    B --> C[Filter incremental window]
    C --> D[Deduplicate by business key]
    D --> E[MERGE into target table]
    E --> F{Target write success?}
    F -- Yes --> G[Update watermark]
    G --> H[Append success run log]
    F -- No --> I[Keep old watermark]
    I --> J[Append failed run log]
    H --> K[Rerun skips already processed window]
    J --> L[Rerun can retry same window]
```

Key idea:

> Watermark ควรขยับหลัง target write สำเร็จเท่านั้น ถ้า target write fail ต้องเก็บ watermark เดิมไว้ เพื่อให้ rerun สามารถ retry window เดิมได้อย่างปลอดภัย


## Setup / imports


In [14]:
from datetime import datetime
from uuid import uuid4

from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

StatementMeta(, b04edc7b-55f7-430a-8c42-c3f4dcfa68ce, 16, Finished, Available, Finished, False)

## Create mock data

Mock data วันนี้เป็น order update records จาก source system เดียวกัน โดยมี `updated_at` เป็น column ที่ใช้ทำ watermark

ตั้งใจใส่เคสสำคัญไว้ 3 แบบ:

- new orders ที่เข้ามาใน batch แรก
- order เดิมที่ถูก update ใน batch ถัดไป
- duplicate business key ใน source window เดียวกัน เพื่อฝึก deduplicate ก่อน merge


In [15]:
# O002 and O003 appear more than once to simulate updates for the same business key.
source_orders_data = [
    ("O001", "C001", "created", 1200.00, "2026-05-01 08:10:00", 1, "crm"),
    ("O002", "C002", "created", 500.00, "2026-05-01 08:20:00", 1, "crm"),
    ("O003", "C003", "created", 850.00, "2026-05-02 09:00:00", 1, "web"),
    ("O003", "C003", "paid", 850.00, "2026-05-02 10:00:00", 2, "web"),
    ("O004", "C004", "created", 2300.00, "2026-05-02 12:00:00", 1, "crm"),
    ("O002", "C002", "cancelled", 500.00, "2026-05-03 11:30:00", 2, "crm"),
    ("O005", "C005", "created", 950.00, "2026-05-03 13:00:00", 1, "web"),
    ("O006", "C006", "created", 1500.00, "2026-05-04 15:10:00", 1, "crm"),
]

source_orders_schema = T.StructType([
    T.StructField("order_id", T.StringType(), False),
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("order_status", T.StringType(), True),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("updated_at_raw", T.StringType(), True),
    T.StructField("source_sequence", T.IntegerType(), True),
    T.StructField("source_system", T.StringType(), True),
])

df_source_orders = (
    spark.createDataFrame(source_orders_data, source_orders_schema)
    .withColumn("updated_at", F.to_timestamp("updated_at_raw"))
    .drop("updated_at_raw")
)

df_source_orders.orderBy("updated_at", "order_id", "source_sequence").show(truncate=False)

StatementMeta(, b04edc7b-55f7-430a-8c42-c3f4dcfa68ce, 17, Finished, Available, Finished, False)

+--------+-----------+------------+------+---------------+-------------+-------------------+
|order_id|customer_id|order_status|amount|source_sequence|source_system|updated_at         |
+--------+-----------+------------+------+---------------+-------------+-------------------+
|O001    |C001       |created     |1200.0|1              |crm          |2026-05-01 08:10:00|
|O002    |C002       |created     |500.0 |1              |crm          |2026-05-01 08:20:00|
|O003    |C003       |created     |850.0 |1              |web          |2026-05-02 09:00:00|
|O003    |C003       |paid        |850.0 |2              |web          |2026-05-02 10:00:00|
|O004    |C004       |created     |2300.0|1              |crm          |2026-05-02 12:00:00|
|O002    |C002       |cancelled   |500.0 |2              |crm          |2026-05-03 11:30:00|
|O005    |C005       |created     |950.0 |1              |web          |2026-05-03 13:00:00|
|O006    |C006       |created     |1500.0|1              |crm         

## PySpark code examples

ใน section นี้เราจะสร้าง Lakehouse demo tables 3 ตาราง:

1. target table สำหรับเก็บ latest order state
2. watermark table สำหรับจำ `last_successful_watermark`
3. run log table สำหรับ audit แต่ละ run

### Example 1: Define demo table names

In [16]:
pipeline_name = "day23_order_incremental_load"

target_table_name = "day23_silver_orders_current"
watermark_table_name = "day23_control_watermark"
run_log_table_name = "day23_audit_incremental_run_log"

print("Target table:", target_table_name)
print("Watermark table:", watermark_table_name)
print("Run log table:", run_log_table_name)

StatementMeta(, b04edc7b-55f7-430a-8c42-c3f4dcfa68ce, 18, Finished, Available, Finished, False)

Target table: day23_silver_orders_current
Watermark table: day23_control_watermark
Run log table: day23_audit_incremental_run_log


### Example 2: Initialize demo tables

In [17]:
# Target table: latest state of each order.
target_schema = T.StructType([
    T.StructField("order_id", T.StringType(), False),
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("order_status", T.StringType(), True),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("updated_at", T.TimestampType(), True),
    T.StructField("source_sequence", T.IntegerType(), True),
    T.StructField("last_run_id", T.StringType(), True),
    T.StructField("processed_at", T.TimestampType(), True),
])

spark.createDataFrame([], target_schema).write.format("delta").mode("overwrite").saveAsTable(target_table_name)

# Watermark table: one row per pipeline in this lab.
watermark_schema = T.StructType([
    T.StructField("pipeline_name", T.StringType(), False),
    T.StructField("watermark_column", T.StringType(), True),
    T.StructField("last_successful_watermark", T.TimestampType(), True),
    T.StructField("last_run_id", T.StringType(), True),
    T.StructField("updated_at", T.TimestampType(), True),
])

initial_watermark_data = [
    (pipeline_name, "updated_at", datetime(1900, 1, 1, 0, 0, 0), None, datetime.now())
]

spark.createDataFrame(initial_watermark_data, watermark_schema).write.format("delta").mode("overwrite").saveAsTable(watermark_table_name)

# Run log table: append one row per run.
run_log_schema = T.StructType([
    T.StructField("run_id", T.StringType(), False),
    T.StructField("pipeline_name", T.StringType(), True),
    T.StructField("batch_label", T.StringType(), True),
    T.StructField("run_status", T.StringType(), True),
    T.StructField("lower_watermark", T.TimestampType(), True),
    T.StructField("upper_watermark", T.TimestampType(), True),
    T.StructField("source_record_count", T.LongType(), True),
    T.StructField("deduped_record_count", T.LongType(), True),
    T.StructField("merge_input_record_count", T.LongType(), True),
    T.StructField("started_at", T.TimestampType(), True),
    T.StructField("completed_at", T.TimestampType(), True),
    T.StructField("note", T.StringType(), True),
])

spark.createDataFrame([], run_log_schema).write.format("delta").mode("overwrite").saveAsTable(run_log_table_name)

print("Day 23 demo tables have been initialized.")

StatementMeta(, b04edc7b-55f7-430a-8c42-c3f4dcfa68ce, 19, Finished, Available, Finished, False)

Day 23 demo tables have been initialized.


### Example 3: Inspect initial watermark

ก่อนเริ่ม incremental load เราต้องรู้ว่า pipeline ประมวลผลสำเร็จถึงจุดไหนแล้ว

รอบแรกจะเริ่มจากค่าเริ่มต้น `1900-01-01 00:00:00` เพื่อให้ดึงข้อมูลชุดแรกได้ทั้งหมดตาม upper watermark ที่กำหนด


In [18]:
spark.table(watermark_table_name).show(truncate=False)

StatementMeta(, b04edc7b-55f7-430a-8c42-c3f4dcfa68ce, 20, Finished, Available, Finished, False)

+----------------------------+----------------+-------------------------+-----------+-------------------------+
|pipeline_name               |watermark_column|last_successful_watermark|last_run_id|updated_at               |
+----------------------------+----------------+-------------------------+-----------+-------------------------+
|day23_order_incremental_load|updated_at      |1900-01-01 00:00:00      |NULL       |2026-06-07 09:41:40.56027|
+----------------------------+----------------+-------------------------+-----------+-------------------------+



### Example 4: Create helper functions for watermark, merge, and run log

วันนี้จะใช้ helper functions ใน notebook เพื่อให้เห็น pipeline flow ชัดขึ้น

สิ่งที่ functions เหล่านี้ทำ:

- อ่าน lower watermark
- deduplicate records ใน incremental window
- merge source batch เข้า target table
- append run log
- update watermark หลัง run สำเร็จ


In [20]:
def get_current_watermark(pipeline_name: str):
    """Return the current last successful watermark for one pipeline."""
    watermark_row = (
        spark.table(watermark_table_name)
        .filter(F.col("pipeline_name") == pipeline_name)
        .select("last_successful_watermark")
        .first()  # Watermark table should have one row per pipeline.
    )
    return watermark_row["last_successful_watermark"]


def append_run_log(
    run_id: str,
    batch_label: str,
    run_status: str,
    lower_watermark,
    upper_watermark,
    source_record_count: int,
    deduped_record_count: int,
    merge_input_record_count: int,
    started_at,
    note: str,
):
    """Append one lightweight audit row for this incremental run."""
    completed_at = datetime.now()
    log_data = [(
        run_id,
        pipeline_name,
        batch_label,
        run_status,
        lower_watermark,
        upper_watermark,
        source_record_count,
        deduped_record_count,
        merge_input_record_count,
        started_at,
        completed_at,
        note,
    )]

    spark.createDataFrame(log_data, run_log_schema).write.format("delta").mode("append").saveAsTable(run_log_table_name)


def update_watermark(pipeline_name: str, new_watermark, run_id: str):
    """Update the watermark only after the target write has completed successfully."""
    watermark_update_data = [(
        pipeline_name,
        "updated_at",
        new_watermark,
        run_id,
        datetime.now(),
    )]

    df_watermark_update = spark.createDataFrame(watermark_update_data, watermark_schema)
    df_watermark_update.createOrReplaceTempView("v_day23_watermark_update")

    spark.sql(f"""
        MERGE INTO {watermark_table_name} AS target
        USING v_day23_watermark_update AS source
        ON target.pipeline_name = source.pipeline_name
        WHEN MATCHED THEN UPDATE SET
            target.watermark_column = source.watermark_column,
            target.last_successful_watermark = source.last_successful_watermark,
            target.last_run_id = source.last_run_id,
            target.updated_at = source.updated_at
        WHEN NOT MATCHED THEN INSERT (
            pipeline_name,
            watermark_column,
            last_successful_watermark,
            last_run_id,
            updated_at
        ) VALUES (
            source.pipeline_name,
            source.watermark_column,
            source.last_successful_watermark,
            source.last_run_id,
            source.updated_at
        )
    """)


def deduplicate_incremental_batch(df_incremental):
    """Keep only the latest record per order_id inside one incremental window."""
    latest_order_window = Window.partitionBy("order_id").orderBy(
        F.col("updated_at").desc(),
        F.col("source_sequence").desc(),
    )

    return (
        df_incremental
        .withColumn("rn", F.row_number().over(latest_order_window))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )


def merge_orders_incremental(df_incremental_latest, run_id: str):
    """Merge incremental records into the current target table."""
    df_merge_input = (
        df_incremental_latest
        .withColumn("last_run_id", F.lit(run_id))
        .withColumn("processed_at", F.current_timestamp())
    )

    df_merge_input.createOrReplaceTempView("v_day23_incremental_orders")

    spark.sql(f"""
        MERGE INTO {target_table_name} AS target
        USING v_day23_incremental_orders AS source
        ON target.order_id = source.order_id
        WHEN MATCHED AND (
            source.updated_at > target.updated_at
            OR (
                source.updated_at = target.updated_at
                AND source.source_sequence > target.source_sequence
            )
        ) THEN UPDATE SET
            target.customer_id = source.customer_id,
            target.order_status = source.order_status,
            target.amount = source.amount,
            target.source_system = source.source_system,
            target.updated_at = source.updated_at,
            target.source_sequence = source.source_sequence,
            target.last_run_id = source.last_run_id,
            target.processed_at = source.processed_at
        WHEN NOT MATCHED THEN INSERT (
            order_id,
            customer_id,
            order_status,
            amount,
            source_system,
            updated_at,
            source_sequence,
            last_run_id,
            processed_at
        ) VALUES (
            source.order_id,
            source.customer_id,
            source.order_status,
            source.amount,
            source.source_system,
            source.updated_at,
            source.source_sequence,
            source.last_run_id,
            source.processed_at
        )
    """)

StatementMeta(, b04edc7b-55f7-430a-8c42-c3f4dcfa68ce, 22, Finished, Available, Finished, False)

### Example 5: Wrap the incremental load into one function

Function นี้จำลอง pipeline run หนึ่งรอบ:

1. อ่าน lower watermark จาก table
2. รับ upper watermark เป็น input
3. filter source ด้วย `lower < updated_at <= upper`
4. deduplicate source batch
5. merge เข้า target
6. update watermark หลัง merge สำเร็จ
7. append run log

ถ้า upper watermark ไม่มากกว่า lower watermark จะถือว่าเป็น rerun window เดิมหรือ window ที่ย้อนกลับ และจะ `SKIPPED`


In [21]:
def run_incremental_load(batch_label: str, upper_watermark_str: str):
    """Run one incremental load window for the Day 23 lab."""
    run_id = str(uuid4())
    started_at = datetime.now()
    lower_watermark = get_current_watermark(pipeline_name)
    upper_watermark = datetime.strptime(upper_watermark_str, "%Y-%m-%d %H:%M:%S")  # Parse watermark string into Python datetime.

    print("=" * 80)
    print("Batch label:", batch_label)
    print("Run ID:", run_id)
    print("Lower watermark:", lower_watermark)
    print("Upper watermark:", upper_watermark)

    if upper_watermark <= lower_watermark:
        note = "Skipped because upper watermark is not greater than lower watermark."
        append_run_log(
            run_id=run_id,
            batch_label=batch_label,
            run_status="SKIPPED",
            lower_watermark=lower_watermark,
            upper_watermark=upper_watermark,
            source_record_count=0,
            deduped_record_count=0,
            merge_input_record_count=0,
            started_at=started_at,
            note=note,
        )
        print(note)
        return {
            "run_id": run_id,
            "run_status": "SKIPPED",
            "source_record_count": 0,
            "deduped_record_count": 0,
        }

    df_incremental_raw = (
        df_source_orders
        .filter(F.col("updated_at") > F.lit(lower_watermark).cast("timestamp"))
        .filter(F.col("updated_at") <= F.lit(upper_watermark).cast("timestamp"))
    )

    source_record_count = df_incremental_raw.count()
    print("Source records in window:", source_record_count)

    df_incremental_latest = deduplicate_incremental_batch(df_incremental_raw)
    deduped_record_count = df_incremental_latest.count()
    print("Records after deduplication:", deduped_record_count)

    df_incremental_latest.orderBy("updated_at", "order_id").show(truncate=False)

    if deduped_record_count > 0:
        merge_orders_incremental(df_incremental_latest, run_id)

    # Update watermark only after the target write step completes or no records need to be merged.
    update_watermark(pipeline_name, upper_watermark, run_id)

    append_run_log(
        run_id=run_id,
        batch_label=batch_label,
        run_status="SUCCESS",
        lower_watermark=lower_watermark,
        upper_watermark=upper_watermark,
        source_record_count=source_record_count,
        deduped_record_count=deduped_record_count,
        merge_input_record_count=deduped_record_count,
        started_at=started_at,
        note="Incremental load completed successfully.",
    )

    print("Incremental load completed successfully.")
    return {
        "run_id": run_id,
        "run_status": "SUCCESS",
        "source_record_count": source_record_count,
        "deduped_record_count": deduped_record_count,
    }

StatementMeta(, b04edc7b-55f7-430a-8c42-c3f4dcfa68ce, 23, Finished, Available, Finished, False)

### Example 6: Run batch 001

Batch แรกใช้ upper watermark ถึง `2026-05-02 23:59:59`

Expected behavior:

- source records ใน window = 5 records
- deduplicate แล้วเหลือ 4 orders เพราะ `O003` มี 2 versions ใน window เดียวกัน
- target table มี 4 orders
- watermark ถูก update เป็น `2026-05-02 23:59:59`


In [22]:
batch_001_result = run_incremental_load("batch_001", "2026-05-02 23:59:59")

spark.table(target_table_name).orderBy("order_id").show(truncate=False)
spark.table(watermark_table_name).show(truncate=False)

StatementMeta(, b04edc7b-55f7-430a-8c42-c3f4dcfa68ce, 24, Finished, Available, Finished, False)

Batch label: batch_001
Run ID: 226099a6-7822-4d89-9230-6933f5981879
Lower watermark: 1900-01-01 00:00:00
Upper watermark: 2026-05-02 23:59:59
Source records in window: 5
Records after deduplication: 4
+--------+-----------+------------+------+---------------+-------------+-------------------+
|order_id|customer_id|order_status|amount|source_sequence|source_system|updated_at         |
+--------+-----------+------------+------+---------------+-------------+-------------------+
|O001    |C001       |created     |1200.0|1              |crm          |2026-05-01 08:10:00|
|O002    |C002       |created     |500.0 |1              |crm          |2026-05-01 08:20:00|
|O003    |C003       |paid        |850.0 |2              |web          |2026-05-02 10:00:00|
|O004    |C004       |created     |2300.0|1              |crm          |2026-05-02 12:00:00|
+--------+-----------+------------+------+---------------+-------------+-------------------+

Incremental load completed successfully.
+--------+---

### Example 7: Rerun batch 001 with the same upper watermark

หลัง batch 001 สำเร็จแล้ว watermark อยู่ที่ `2026-05-02 23:59:59`

ถ้าเรารัน window เดิมซ้ำ function จะ `SKIPPED` เพราะ upper watermark ไม่มากกว่า lower watermark

นี่คือ rerun safety แบบหนึ่ง: pipeline ไม่ย้อนกลับไป process window เดิมซ้ำโดยไม่ตั้งใจ


In [23]:
batch_001_rerun_result = run_incremental_load("batch_001_rerun", "2026-05-02 23:59:59")

spark.table(target_table_name).orderBy("order_id").show(truncate=False)
spark.table(run_log_table_name).orderBy("started_at").show(truncate=False)

StatementMeta(, b04edc7b-55f7-430a-8c42-c3f4dcfa68ce, 25, Finished, Available, Finished, False)

Batch label: batch_001_rerun
Run ID: c9162e37-c805-4916-a69b-b4f24e446c68
Lower watermark: 2026-05-02 23:59:59
Upper watermark: 2026-05-02 23:59:59
Skipped because upper watermark is not greater than lower watermark.
+--------+-----------+------------+------+-------------+-------------------+---------------+------------------------------------+--------------------------+
|order_id|customer_id|order_status|amount|source_system|updated_at         |source_sequence|last_run_id                         |processed_at              |
+--------+-----------+------------+------+-------------+-------------------+---------------+------------------------------------+--------------------------+
|O001    |C001       |created     |1200.0|crm          |2026-05-01 08:10:00|1              |226099a6-7822-4d89-9230-6933f5981879|2026-06-07 09:42:51.451645|
|O002    |C002       |created     |500.0 |crm          |2026-05-01 08:20:00|1              |226099a6-7822-4d89-9230-6933f5981879|2026-06-07 09:42:51.451645

### Example 8: Run batch 002 for the next incremental window

Batch ถัดไปใช้ upper watermark ถึง `2026-05-04 23:59:59`

Expected behavior:

- ดึงเฉพาะ records ที่ `updated_at` มากกว่า watermark เดิมและไม่เกิน upper watermark ใหม่
- `O002` ถูก update จาก `created` เป็น `cancelled`
- `O005` และ `O006` ถูก insert ใหม่
- target table มี 6 unique orders


In [24]:
batch_002_result = run_incremental_load("batch_002", "2026-05-04 23:59:59")

spark.table(target_table_name).orderBy("order_id").show(truncate=False)
spark.table(watermark_table_name).show(truncate=False)

StatementMeta(, b04edc7b-55f7-430a-8c42-c3f4dcfa68ce, 26, Finished, Available, Finished, False)

Batch label: batch_002
Run ID: a1331a96-6e02-41e8-b86d-627b46bd2aeb
Lower watermark: 2026-05-02 23:59:59
Upper watermark: 2026-05-04 23:59:59
Source records in window: 3
Records after deduplication: 3
+--------+-----------+------------+------+---------------+-------------+-------------------+
|order_id|customer_id|order_status|amount|source_sequence|source_system|updated_at         |
+--------+-----------+------------+------+---------------+-------------+-------------------+
|O002    |C002       |cancelled   |500.0 |2              |crm          |2026-05-03 11:30:00|
|O005    |C005       |created     |950.0 |1              |web          |2026-05-03 13:00:00|
|O006    |C006       |created     |1500.0|1              |crm          |2026-05-04 15:10:00|
+--------+-----------+------------+------+---------------+-------------+-------------------+

Incremental load completed successfully.
+--------+-----------+------------+------+-------------+-------------------+---------------+--------------

### Example 9: Validate idempotency with duplicate check

หลัง incremental load หลายรอบ target table ควรมี key เดียวต่อหนึ่ง order เท่านั้น

ถ้ามี duplicate `order_id` แปลว่า write pattern อาจยังไม่ idempotent หรือ merge key ไม่ถูกต้อง


In [25]:
df_duplicate_orders = (
    spark.table(target_table_name)
    .groupBy("order_id")
    .agg(F.count("*").alias("record_count"))
    .filter(F.col("record_count") > 1)
)

df_duplicate_orders.show(truncate=False)
print("Duplicate order_id count:", df_duplicate_orders.count())

StatementMeta(, b04edc7b-55f7-430a-8c42-c3f4dcfa68ce, 27, Finished, Available, Finished, False)

+--------+------------+
|order_id|record_count|
+--------+------------+
+--------+------------+

Duplicate order_id count: 0


### Example 10: Inspect run log

Run log ช่วยตอบคำถามแบบ production ได้ เช่น:

- run ไหนสำเร็จหรือถูก skip
- run ไหนอ่าน source มากี่ records
- run ไหน deduplicate แล้วเหลือกี่ records
- watermark ของ run นั้นเริ่มและจบตรงไหน


In [26]:
spark.table(run_log_table_name).orderBy("started_at").show(truncate=False)

StatementMeta(, b04edc7b-55f7-430a-8c42-c3f4dcfa68ce, 28, Finished, Available, Finished, False)

+------------------------------------+----------------------------+---------------+----------+-------------------+-------------------+-------------------+--------------------+------------------------+--------------------------+--------------------------+--------------------------------------------------------------------+
|run_id                              |pipeline_name               |batch_label    |run_status|lower_watermark    |upper_watermark    |source_record_count|deduped_record_count|merge_input_record_count|started_at                |completed_at              |note                                                                |
+------------------------------------+----------------------------+---------------+----------+-------------------+-------------------+-------------------+--------------------+------------------------+--------------------------+--------------------------+--------------------------------------------------------------------+
|226099a6-7822-4d89-9230-693

## Quick recap

| Question | Answer |
|---|---|
| Watermark ใช้ทำอะไร? | ใช้จำว่า pipeline process สำเร็จถึงจุดไหนแล้ว |
| ทำไมต้องมี upper watermark? | เพื่อให้ batch window ชัดเจน และไม่อ่านข้อมูลที่กำลังเปลี่ยนระหว่าง run แบบไร้ขอบเขต |
| ทำไมไม่ควร update watermark ก่อน write target สำเร็จ? | ถ้า write fail แต่ watermark เลื่อนไปแล้ว อาจทำให้ข้อมูลบางช่วงถูกข้าม |
| Idempotency แปลว่าอะไรใน pipeline? | รันซ้ำแล้ว target table ยังถูกต้อง ไม่ duplicate และไม่ corrupt |
| ทำไมใช้ `MERGE` แทน append ตรง ๆ? | เพราะ business key เดิมควรถูก update ไม่ใช่ insert ซ้ำ |
| ทำไมต้อง deduplicate source batch ก่อน merge? | เพื่อกัน source ส่ง key เดียวกันหลาย version ใน window เดียวกัน |


## Exercises

### Exercise 1: Validate the final target table

ตรวจว่า target table หลัง batch 001 + batch 002 มี expected state ตามนี้:

Requirements:

- target table มี 6 unique orders
- ไม่มี duplicate `order_id`
- `O002` มี `order_status = cancelled`
- `O003` ใช้ version ล่าสุดใน batch แรก คือ `paid`

Expected result:

- `target_count = 6`
- `duplicate_key_count = 0`
- เห็น `O002` เป็น `cancelled` และ `O003` เป็น `paid`


In [27]:
df_target_final = spark.table(target_table_name)

target_count = df_target_final.count()
duplicate_key_count = (
    df_target_final
    .groupBy("order_id")
    .agg(F.count("*").alias("record_count"))
    .filter(F.col("record_count") > 1)
    .count()
)

print("Target count:", target_count)
print("Duplicate key count:", duplicate_key_count)

(
    df_target_final
    .filter(F.col("order_id").isin("O002", "O003"))
    .select("order_id", "order_status", "updated_at", "source_sequence")
    .orderBy("order_id")
    .show(truncate=False)
)

StatementMeta(, b04edc7b-55f7-430a-8c42-c3f4dcfa68ce, 29, Finished, Available, Finished, False)

Target count: 6
Duplicate key count: 0
+--------+------------+-------------------+---------------+
|order_id|order_status|updated_at         |source_sequence|
+--------+------------+-------------------+---------------+
|O002    |cancelled   |2026-05-03 11:30:00|2              |
|O003    |paid        |2026-05-02 10:00:00|2              |
+--------+------------+-------------------+---------------+



### Exercise 2: Rerun the latest window and confirm target count is unchanged

ลองรันด้วย upper watermark เดิมอีกครั้ง เพื่อยืนยันว่า function ไม่ย้อนกลับไป process window เดิม

Requirements:

- เก็บ target count ก่อน rerun
- run incremental load ด้วย upper watermark `2026-05-04 23:59:59`
- เก็บ target count หลัง rerun
- ตรวจว่า count เท่าเดิม

Expected result:

- run status เป็น `SKIPPED`
- target count ก่อนและหลัง rerun เท่ากัน


In [28]:
target_count_before_rerun = spark.table(target_table_name).count()

exercise_rerun_result = run_incremental_load("exercise_same_upper_watermark", "2026-05-04 23:59:59")

target_count_after_rerun = spark.table(target_table_name).count()

print("Exercise rerun result:", exercise_rerun_result)
print("Target count before rerun:", target_count_before_rerun)
print("Target count after rerun:", target_count_after_rerun)
print("Count unchanged:", target_count_before_rerun == target_count_after_rerun)

StatementMeta(, b04edc7b-55f7-430a-8c42-c3f4dcfa68ce, 30, Finished, Available, Finished, False)

Batch label: exercise_same_upper_watermark
Run ID: 77e3591e-a633-4c32-9210-04f0d5b60e7d
Lower watermark: 2026-05-04 23:59:59
Upper watermark: 2026-05-04 23:59:59
Skipped because upper watermark is not greater than lower watermark.
Exercise rerun result: {'run_id': '77e3591e-a633-4c32-9210-04f0d5b60e7d', 'run_status': 'SKIPPED', 'source_record_count': 0, 'deduped_record_count': 0}
Target count before rerun: 6
Target count after rerun: 6
Count unchanged: True


### Exercise 3: Simulate a late-arriving record risk

Watermark ด้วย `updated_at` มีข้อควรระวัง: ถ้า record เพิ่งถูกส่งมาถึง pipeline แต่ค่า `updated_at` เก่ากว่า current watermark แล้ว logic ปกติอย่าง `updated_at > current_watermark` จะไม่ดึง record นี้เข้ามาประมวลผล

Requirements:

- สร้าง late-arriving record 1 แถวที่ `updated_at = 2026-05-03 08:00:00`
- อ่าน current watermark จาก table
- filter ด้วย logic ปกติ `updated_at > current_watermark`
- ตรวจว่าถูกจับได้หรือไม่

Expected result:

- record นี้ไม่ถูกจับโดย normal watermark filter
- ได้ learning note ว่า production อาจต้องมี lookback window, backfill, หรือ late-arrival handling เพิ่มเติม


In [29]:
late_arriving_data = [
    ("O007", "C007", "created", 700.00, "2026-05-03 08:00:00", 1, "external_partner"),
]

df_late_arriving_orders = (
    spark.createDataFrame(late_arriving_data, source_orders_schema)
    .withColumn("updated_at", F.to_timestamp("updated_at_raw"))
    .drop("updated_at_raw")
)

current_watermark = get_current_watermark(pipeline_name)

print("Current watermark:", current_watermark)

df_late_captured_by_normal_filter = (
    df_late_arriving_orders
    .filter(F.col("updated_at") > F.lit(current_watermark).cast("timestamp"))
)

df_late_arriving_orders.show(truncate=False)
df_late_captured_by_normal_filter.show(truncate=False)

print("Late-arriving rows captured by normal watermark filter:", df_late_captured_by_normal_filter.count())
print("Learning note: Pure updated_at watermark logic may miss late-arriving records.")

StatementMeta(, b04edc7b-55f7-430a-8c42-c3f4dcfa68ce, 31, Finished, Available, Finished, False)

Current watermark: 2026-05-04 23:59:59
+--------+-----------+------------+------+---------------+----------------+-------------------+
|order_id|customer_id|order_status|amount|source_sequence|source_system   |updated_at         |
+--------+-----------+------------+------+---------------+----------------+-------------------+
|O007    |C007       |created     |700.0 |1              |external_partner|2026-05-03 08:00:00|
+--------+-----------+------------+------+---------------+----------------+-------------------+

+--------+-----------+------------+------+---------------+-------------+----------+
|order_id|customer_id|order_status|amount|source_sequence|source_system|updated_at|
+--------+-----------+------------+------+---------------+-------------+----------+
+--------+-----------+------------+------+---------------+-------------+----------+

Late-arriving rows captured by normal watermark filter: 0
Learning note: Pure updated_at watermark logic may miss late-arriving records.


## Common mistakes

1. **Update watermark ก่อน target write สำเร็จ**

   ถ้า update watermark แล้ว merge/write fail ข้อมูลในช่วงนั้นอาจถูกข้ามในรอบถัดไป ดังนั้นควร update watermark หลัง write สำเร็จแล้วเท่านั้น

2. **ใช้ append อย่างเดียวกับ incremental update**

   ถ้า source ส่ง record เดิมที่เปลี่ยนสถานะมาอีกครั้ง การ append ตรง ๆ จะทำให้เกิด duplicate ทาง business key ดังนั้นควรใช้ `MERGE` หรือ upsert pattern ที่อิง business key

3. **ไม่ deduplicate source batch ก่อน merge**

   ถ้า source window มี key เดียวกันหลาย version อาจทำให้เกิด `MERGE` ambiguity เพราะ source หลายแถว match กับ target key เดียวกัน ดังนั้นควรเลือก record ล่าสุดต่อ key ก่อน merge

4. **ใช้แค่ timestamp โดยไม่มี tie-breaker**

   ถ้า records หลายแถวมี `updated_at` เท่ากัน ควรมี tie-breaker เช่น `source_sequence`, `ingestion_timestamp`, หรือ source offset เพื่อเลือก record ล่าสุดให้ชัด

5. **ใช้ `>= lower_watermark` โดยไม่คิดเรื่อง rerun**

   การใช้ `>= lower_watermark` อาจทำให้ pipeline อ่าน record จาก source ที่มี `updated_at` เท่ากับ watermark เดิมกลับมา process ซ้ำ ถ้า write เป็น append จะ duplicate ได้ง่าย ดังนั้น lab นี้จึงใช้ `>` สำหรับ lower bound และ `<=` สำหรับ upper bound

6. **ไม่มี run log**

   ถ้าไม่มี run log จะตอบยากว่า run ไหนอ่านกี่ records, fail ตรงไหน, watermark ก่อน/หลังคืออะไร และควร retry จากจุดไหน

7. **ไม่คิด late-arriving records**

   Record ที่มาช้าแต่มี `updated_at` เก่ากว่า watermark อาจถูกข้ามได้ ใน production อาจต้องมี lookback window, backfill process, หรือ late-arrival policy

8. **ขยับ watermark ถอยหลังโดยไม่ตั้งใจ**

   Watermark ควร move forward อย่างควบคุมได้ ถ้าเลื่อนถอยหลังโดยไม่ตั้งใจ pipeline อาจ reprocess ข้อมูลเก่าเยอะหรือเกิด duplicate downstream


## Expected Output / Success Criteria

เมื่อจบ Day 23 ควรทำได้:

- อธิบายได้ว่า watermark คืออะไร และต่างจาก run log อย่างไร
- สร้าง mock source data ที่มี `updated_at` สำหรับ incremental load ได้
- สร้าง target table, watermark table และ run log table แบบง่ายใน Fabric Lakehouse ได้
- filter incremental window ด้วย pattern `lower_watermark < updated_at <= upper_watermark` ได้
- deduplicate source batch ด้วย `row_number()` ก่อน merge ได้
- ใช้ `MERGE` เพื่อ update/insert records เข้า target table ได้
- update watermark หลัง successful write ได้
- rerun window เดิมแล้ว target count ไม่เพิ่มผิดปกติ
- ตรวจ duplicate business key ใน target table ได้
- อธิบายความเสี่ยงของ late-arriving records ได้

## Final takeaway

Incremental load ที่ไว้ใจได้ต้องคิดเป็น flow ทั้งหมด ไม่ใช่แค่เขียน filter ด้วย watermark แล้วจบ

> Watermark บอกว่าอ่านถึงไหนแล้ว  
> MERGE ทำให้ write ปลอดภัยขึ้น  
> Run log ทำให้ตรวจสอบและ retry ได้  
> Idempotency ทำให้รันซ้ำแล้ว target ยังถูกต้อง

จำ mindset นี้ไว้:

- อย่า update watermark ก่อน target write สำเร็จ
- อย่า append records ที่เป็น update ของ business key เดิมโดยตรง
- อย่าลืม deduplicate source window ให้เหลือ latest record ต่อ key ก่อน merge
- อย่าคิดว่า `updated_at > watermark` แก้ทุก edge case ได้
- ทุก incremental pipeline ควรตอบได้ว่า “run นี้อ่านจากไหนถึงไหน และเขียนอะไรไปบ้าง”


## Optional cleanup

In [13]:
# spark.sql(f"DROP TABLE IF EXISTS {target_table_name}")
# spark.sql(f"DROP TABLE IF EXISTS {watermark_table_name}")
# spark.sql(f"DROP TABLE IF EXISTS {run_log_table_name}")

StatementMeta(, b04edc7b-55f7-430a-8c42-c3f4dcfa68ce, 15, Finished, Available, Finished, False)

DataFrame[]